# Estatística descritiva — análise monovariada

**Capítulo 21** do livro vivo [Ciência de Dados e Aprendizado de Máquina](https://machinelearning.ghdaru.com.br/21-analise-exploratoria.html).

O caminho da aula, na ordem: **tipo de cada campo → contagem e nulidade → medidas de posição → medidas separatrizes → histograma e boxplot**, uma variável de cada vez.

Ao final você vai ter encontrado, com o próprio código, as duas colunas em que a leitura automática **engana**.

> Este notebook usa **pandas** e **matplotlib** — as duas já vêm no Colab. É a única etapa da trilha que sai da biblioteca padrão, e a razão está no [ADR 0010](https://github.com/GHDaru/machinelearning/blob/main/adr/0010-pandas-na-etapa-de-exploracao.md): desenhar histograma e boxplot à mão ensinaria sobre desenho, não sobre distribuição.

In [ ]:
# --- roda igual na sua máquina e no Colab ------------------------------
import pathlib, urllib.request

RAW = "https://raw.githubusercontent.com/GHDaru/machinelearning/main/"
REL = "ml-zero/dados/limonada/limonada.csv"

raiz = pathlib.Path.cwd()
for _ in range(5):
    if (raiz / "ml-zero").is_dir():
        break
    raiz = raiz.parent
else:
    raiz = pathlib.Path.cwd()

CSV = raiz / REL
if not CSV.exists():
    CSV.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(RAW + REL, CSV)
    print("baixado.")
print(CSV)

## 1. Que tipo é cada campo?

Antes de qualquer estatística: **o que cada coluna é**. Número que é medida, número que é contagem, número que na verdade é categoria, texto, data.

In [ ]:
import pandas as pd

df = pd.read_csv(CSV, parse_dates=["data"])
print(df.dtypes)
print()
print("linhas:", len(df))
df.head()

`dtypes` diz como o pandas **leu**, não o que a coluna **é**. Duas checagens rápidas separam as duas coisas:

In [ ]:
for col in df.columns:
    print(f"{col:14s} {str(df[col].dtype):16s} distintos={df[col].nunique():4d}  nulos={df[col].isna().sum()}")

Repare em `preco`: é `float64`, e tem **2 valores distintos** em 365 linhas. É um número que se comporta como categoria — e isso vai importar daqui a pouco.

E repare que **nenhuma coluna tem nulos**. Isso não é atestado de qualidade: pode significar que dias sem operação simplesmente não viraram linha.

## 2. Contagem e nulidade

O `describe()` é o atalho, e vale saber o que ele **não** mostra: moda, e nada sobre a forma da distribuição.

In [ ]:
df.describe()

## 3. Medidas de posição

**Tendência central:** média, mediana e moda.

- a **média** é sensível: um valor extremo a puxa;
- a **mediana** é insensível: só a posição importa;
- a **moda** é o valor mais frequente — e no varejo é ela que responde "qual preço o cliente vê com mais frequência?".

In [ ]:
NUM = ["temperatura", "precipitacao", "panfletos", "preco", "vendas"]

pos = pd.DataFrame({
    "média":   df[NUM].mean(),
    "mediana": df[NUM].median(),
    "moda":    df[NUM].mode().iloc[0],
})
pos["média − mediana"] = pos["média"] - pos["mediana"]
pos.round(3)

A coluna `média − mediana` é o **detector de assimetria mais barato que existe**:

- positiva → cauda à **direita** (a média foi puxada para cima);
- negativa → cauda à **esquerda**;
- perto de zero → aproximadamente simétrica.

## 4. Medidas separatrizes

Percentil, decil e quartil são a **mesma** ideia com nomes diferentes: cortar a distribuição ordenada em partes iguais.

E a mediana é os três ao mesmo tempo — **P50 = Q2 = D5**. Confira:

In [ ]:
col = "temperatura"
p50 = df[col].quantile(0.50)
print(f"P50 = {p50:.2f}   Q2 = {df[col].quantile(2/4):.2f}   D5 = {df[col].quantile(5/10):.2f}   mediana = {df[col].median():.2f}")

print()
print("decis de", col)
print(df[col].quantile([i/10 for i in range(1, 10)]).round(2).to_string())

## 5. Histograma e boxplot

O histograma mostra a **forma**. O boxplot mostra **posição, dispersão e pontas** — e é o único dos dois que marca os candidatos a outlier.

Os cinco elementos do boxplot, na ordem em que se lê:

| Elemento | O que é |
|---|---|
| caixa | de Q1 a Q3 — os **50% do meio** |
| traço dentro da caixa | a **mediana** (P50) |
| bigodes | até o ponto mais extremo **dentro** da cerca |
| cerca | Q1 − 1,5 × IQR e Q3 + 1,5 × IQR |
| pontos soltos | o que caiu fora da cerca |

In [ ]:
import matplotlib
matplotlib.use("Agg")          # no Colab pode remover esta linha
import matplotlib.pyplot as plt

col = "temperatura"
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1]})
ax1.hist(df[col], bins=15, color="#2f6f9f", alpha=.75)
ax1.axvline(df[col].mean(),   color="#e0a24a", ls="--", lw=2, label="média")
ax1.axvline(df[col].median(), color="#2e8b57", ls="--", lw=2, label="mediana")
ax1.legend(); ax1.set_title(f"{col} — histograma e boxplot")
ax2.boxplot(df[col], orientation="horizontal", widths=.6)   # matplotlib < 3.11: use vert=False
ax2.set_xlabel(col)
plt.tight_layout()
plt.savefig("temperatura.png", dpi=90)
print("gráfico salvo em temperatura.png")

## 6. A cerca de 1,5 × IQR, coluna a coluna

Agora a conta que o boxplot faz por dentro, para **todas** as colunas de uma vez.

In [ ]:
def cerca(s):
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    fora = s[(s < li) | (s > ls)]
    return pd.Series({"Q1": q1, "Q3": q3, "IQR": iqr,
                      "cerca_inf": li, "cerca_sup": ls, "fora_da_cerca": len(fora)})

pd.DataFrame({c: cerca(df[c]) for c in NUM}).T.round(3)

## 7. As duas surpresas

**`preco` acusa 62 pontos fora da cerca.** Olhe o IQR dessa coluna: **zero**. Mais de 75% dos dias custam 0,30, então Q1 = Q3 = 0,30 e a cerca tem largura nenhuma. Qualquer valor diferente de 0,30 é acusado — e aumentar o fator de 1,5 para 3,0 não muda nada, porque três vezes zero continua zero.

Os 62 dias são **julho e agosto**: o preço de alta temporada. Removê-los apagaria dois dos meses mais importantes do negócio.

**`precipitacao` acusa 28 pontos.** Aqui não há defeito de régua: a distribuição é **assimétrica à direita**, e a regra de 1,5 × IQR foi pensada para distribuições aproximadamente simétricas. Ela acusa muito em cauda longa **por construção**. São dias de chuva forte — fenômeno, não erro.

In [ ]:
chuva = df["precipitacao"]
print("média  ", round(chuva.mean(), 3))
print("mediana", round(chuva.median(), 3))
print("→ média maior que a mediana: cauda à direita")
print()
print("preco: valores distintos e contagem")
print(df["preco"].value_counts().to_string())
print()
print("meses em que cada preço aparece:")
print(df.assign(mes=df.data.dt.month).groupby("preco").mes.unique().to_string())

## O que levar

- **Tipo declarado não é tipo real.** `preco` é `float64` e se comporta como categoria.
- **`média − mediana` é o detector de assimetria mais barato.** Positivo, cauda à direita.
- **Mediana descreve melhor o que é assimétrico.** "Num dia típico chove 0,74" é verdade; "chove em média 0,83" descreve um dia que quase não existe.
- **A cerca de 1,5 × IQR é um critério, não uma verdade.** Ela quebra em variável quase constante e acusa demais em distribuição assimétrica.
- **Fora da cerca ≠ errado.** A decisão — manter, marcar, transformar, remover — precisa de critério declarado.

Continue no capítulo [21 — Análise Exploratória](https://machinelearning.ghdaru.com.br/21-analise-exploratoria.html), onde o laboratório `21-l1` faz isto no navegador, e os exercícios `21-e4` e `21-e5` cobram as duas surpresas acima.